# 06. Compare fixed model predictions

This notebook explains a reusable, score-only evaluation process. It reads the saved predictions from notebooks 03 and 05; it does not fit models, select features, tune the blend, or change the predictor/simulator.

Both periods are development evidence. The blend's validation predictions use the same validation period that selected its weight, and the existing test period has already influenced project development. This run is online Elo evaluation, not an untouched final benchmark, a periodic-retraining experiment, or a frozen quarterfinal forecast.


## 01. Locate the project and evaluation module

The reusable implementation lives in `src/evaluation.py` and `src/evaluate.py`. Keeping scoring outside the training notebooks lets future model versions use the same population, metric definitions, fallback rules, and reporting process.


In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import Image, display

base = Path.cwd()
if not (base / "notebooks").is_dir():
    base = base.parent
if not (base / "src" / "evaluate.py").is_file():
    raise FileNotFoundError("Start this notebook from the project root or notebooks folder.")
if str(base) not in sys.path:
    sys.path.insert(0, str(base))
from src.evaluate import run_evaluation, file_hash
print("Project:", base)


## 02. Run a fixed evaluation protocol

Use ten calibration bins and 1,000 paired UTC-week bootstrap repetitions with seed 2026 and 95% percentile intervals. These are declared diagnostic settings, not values selected to make a model look better.

The adapter checks canonical match identities/outcomes, exact saved split boundaries, model-input fingerprints, and fallback arithmetic. Every model must predict every match. Unknown regional inputs cannot silently disappear. Inputs, code, configuration, and library versions identify an immutable report folder; identical reruns reuse it.


In [ ]:
run_dir = run_evaluation(base, resamples=1000, seed=2026, confidence=0.95, bins=10)
manifest = json.loads((run_dir / "manifest.json").read_text(encoding="utf-8"))
registry = pd.DataFrame(json.loads((run_dir / "model_registry.json").read_text(encoding="utf-8")))
print("Evaluation:", manifest["run_id"])
print("Status:", manifest["status"])
print("Original notebook metrics reproduced:", manifest["source_metrics_reproduced"])
print("Report:", run_dir / "report.md")


## 03. Inspect model versions and information timing

Compare a constant 50% probability baseline, raw global Elo, global logistic, regional logistic, and the validation-selected blend. The constant baseline uses the common `p >= 0.5` decision rule, so its accuracy equals A's observed win frequency rather than necessarily 50%.

Coefficient-training endpoints precede evaluation information cutoffs. Blend selection reuse is explicitly allowed only for the labeled validation diagnostic. Logical prediction timestamps refer to retrospective pre-match information times, not proof that forecasts were logged before games occurred.


In [ ]:
display(registry[["model_id", "label", "version", "features", "fallback_model_id", "selection_period"]])
display(pd.DataFrame(manifest["periods"]).T)
print("Blend selection used international validation outcomes through:", manifest["blend_selection_end"])
print(manifest["prediction_timestamp_semantics"])
print("Evaluation mode:", manifest["identity"]["configuration"]["evaluation_mode"])


## 04. Compare identical populations and coverage

Full scope retains every game using the model's declared fallback. Common-native scope requires native predictions from every registered model; for this suite, that means eligible international regional inputs. It is a different population from notebook 04's broader raw-local-Elo known-home subset.

Accuracy and ROC AUC are higher-is-better; log loss and Brier score are lower-is-better. ECE summarizes fixed-bin calibration gaps and is sensitive to binning/sample size. Read counts and coverage before interpreting any score difference. The validation blend's apparent score is selection-reused.


In [ ]:
metrics = pd.read_csv(run_dir / "metrics.csv")
coverage = pd.read_csv(run_dir / "coverage.csv")
primary = metrics.loc[metrics["population"].eq("full_scope") & metrics["scope"].isin(["all", "international", "worlds"])]
display(primary[["period", "scope", "model_id", "matches", "accuracy", "log_loss", "brier", "roc_auc", "selection_reused"]])
display(coverage.loc[coverage["population"].eq("full_scope") & coverage["model_id"].eq("blend_v1")])
display(metrics.loc[metrics["population"].eq("common_native") & metrics["scope"].eq("all"),
                    ["period", "model_id", "matches", "accuracy", "log_loss"]])
display(Image(filename=str(run_dir / "metrics.png")))


## 05. Inspect probability calibration and bin counts

A well-calibrated group predicted near 70% should win about 70% of the time over enough comparable observations. Curves near the diagonal indicate agreement, but sparse bins can fluctuate. Empty bins remain explicit in the saved table.

The figure separates validation and development-test periods. The validation blend curve is not independent of weight selection. Calibration here evaluates probability confidence, not the regional pool-strength calibration algorithm.


In [ ]:
calibration = pd.read_csv(run_dir / "calibration.csv")
display(Image(filename=str(run_dir / "calibration.png")))
display(calibration.loc[calibration["period"].eq("development_test") & calibration["scope"].eq("international")
                        & calibration["population"].eq("full_scope") & calibration["matches"].gt(0)
                        & calibration["model_id"].isin(["global_logistic_v1", "blend_v1"]),
                        ["model_id", "lower", "upper", "matches", "predicted_probability", "observed_win_rate"]])


## 06. Read paired differences and approximate uncertainty

Differences are candidate minus global logistic on exactly the same games. Negative log loss/Brier and positive accuracy differences favor the candidate. Whole observed UTC weeks, starting Monday, are resampled in pairs; variable-sized blocks are combined with game-weighted means.

Verified series IDs are unavailable, so weeks are a time-block approximation, not invented series or tournament identities. Weeks may still be dependent and events may cross weeks. Fewer than two blocks gives no interval; fewer than ten is flagged. Intervals condition on fixed predictions and omit model-fitting/selection uncertainty. A range crossing zero signals uncertainty under this approximation; it is not an automated model-selection test.


In [ ]:
differences = pd.read_csv(run_dir / "paired_differences.csv")
display(differences.loc[differences["period"].eq("development_test") & differences["population"].eq("full_scope")
                        & differences["model_id"].eq("blend_v1"),
                        ["scope", "metric", "candidate_minus_baseline", "ci_lower", "ci_upper", "matches", "blocks", "interval_status"]])
display(Image(filename=str(run_dir / "paired_differences.png")))


## 07. Check changes across time

Calendar-year summaries show whether aggregate scores hide uneven behavior across time. These are segments within the saved evaluation periods, not balanced seasons or newly trained walk-forward folds. Training endpoints and sample sizes can differ between the validation and development-test periods.

Do not use this table to repeatedly tune against the same inspected test period. Future historical-statistics models should be developed on validation and compared under this same prediction contract; later untouched data is still needed for final claims.


In [ ]:
temporal = pd.read_csv(run_dir / "temporal_metrics.csv")
display(temporal.loc[temporal["scope"].isin(["international", "worlds"])
                    & temporal["model_id"].isin(["global_logistic_v1", "blend_v1"]),
                    ["period", "scope", "calendar_year", "model_id", "matches", "accuracy", "log_loss", "brier", "native_predictions", "fallback_predictions"]])


## 08. Verify the saved record and plan future comparisons

The normalized ledger preserves each model/match probability, logical forecast and training/selection times, native availability, fallback, and evaluation mode. Match identities and outcomes are stored separately. Gzip keeps the repeated prediction ledger compact.

Output hashes protect saved runs from accidental edits. A new model should receive a new ID/version and a complete prediction ledger on the same canonical matches with explicit fallback. Scoring a frozen pre-tournament forecast requires separately prepared cutoff-safe predictions; the current adapter does not invent brackets, train new folds, or create an untouched period.


In [ ]:
for name, expected in manifest["output_sha256"].items():
    assert file_hash(run_dir / name) == expected, f"Changed output: {name}"
match_ledger = pd.read_csv(run_dir / "matches.csv.gz")
prediction_ledger = pd.read_csv(run_dir / "predictions.csv.gz")
assert len(prediction_ledger) == len(match_ledger) * len(registry)
assert not prediction_ledger.duplicated(["model_id", "match_id"]).any()
print("Canonical games:", len(match_ledger))
print("Prediction rows:", len(prediction_ledger))
print("Registered models:", len(registry))
print("All output hashes verified. No training or model-selection step was performed.")
